# 构建用于强化学习的 Gym 环境（纯 PnL 版本）

本 notebook 将基于 hftbacktest 框架和 PMM 策略构建一个自定义的 OpenAI Gym 环境，用于训练强化学习智能体进行市商策略优化。

## 环境设计目标

-   **状态空间**: 订单簿特征、价格变化、持仓状态、PnL 等
-   **动作空间**: 价差调整、订单数量、策略参数调整等
-   **奖励函数**: 基于 PnL 变化、风险控制等多维度奖励设计
-   **环境**: 基于真实市场数据的高保真模拟环境
-   **设计理念**: 完全遵循 HFTBacktest 原生设计，PnL 从 0 开始


In [1]:

import torch
import warnings

# TorchRL相关导入
from tensordict import TensorDict

# 设置警告过滤
warnings.filterwarnings('ignore')

# 设置默认设备（优先级：CUDA > MPS > CPU）
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("🚀 使用CUDA GPU加速")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🍎 使用Apple Silicon MPS加速")
else:
    device = torch.device("cpu")
    print("💻 使用CPU")

print(f"环境依赖库导入完成，使用设备: {device}")

🍎 使用Apple Silicon MPS加速
环境依赖库导入完成，使用设备: mps


In [2]:
# 测试 PMM 强化学习环境

# 导入我们构建的PMM环境
from lib.rl_env import create_pmm_env
from hftbacktest import BacktestAsset
import numpy as np
import os

# 使用单个测试数据文件
# test_data_file = "data/output/xrpusdt_20250717.npz"
test_data_file = "data/slices/xrpusdt_20250717_2h_77562432/split_003_2h.npz"

if not os.path.exists(test_data_file):
    raise FileNotFoundError(
        f"❌ 测试数据文件不存在: {test_data_file}\n   请确保数据文件存在，或修改 test_data_file 路径")

print(f"✅ 找到测试数据文件: {test_data_file}")

try:
    print(f"🔄 正在加载数据: {test_data_file}")

    # 重要：先加载numpy数组，然后传给BacktestAsset
    test_data = np.load(test_data_file)['data']
    print(f"✅ 数据加载成功，形状: {test_data.shape}")

    # 按照03_strategy_design.ipynb中的正确方式创建BacktestAsset
    data_asset = (
        BacktestAsset()
        .data([test_data])  # 传入numpy数组，而不是文件路径！
        .linear_asset(1.0)       # 线性资产，合约乘数为1
        .risk_adverse_queue_model()  # 风险规避队列模型
        .no_partial_fill_exchange()  # 不允许部分成交
        .constant_latency(10_000_000, 10_000_000)  # 添加延迟设置
        .tick_size(0.0001)         # XRP最小价格精度 (0.0001 USDT)
        .lot_size(0.1)         # XRP最小交易数量
        .trading_value_fee_model(-0.00003, 0.0007)  # Binance手续费模型
        .power_prob_queue_model3(3.0)  # 添加队列模型
    )

    print(f"✅ 成功创建BacktestAsset")
    print(f"   数据形状: {test_data.shape}")
    print(f"   配置完成")
    print(f"   📌 使用纯 PnL 环境（HFTBacktest 原生设计）")

except Exception as e:
    raise RuntimeError(f"❌ 创建BacktestAsset失败: {e}") from e

✅ 找到测试数据文件: data/slices/xrpusdt_20250717_2h_77562432/split_003_2h.npz
🔄 正在加载数据: data/slices/xrpusdt_20250717_2h_77562432/split_003_2h.npz
✅ 数据加载成功，形状: (5550320,)
✅ 成功创建BacktestAsset
   数据形状: (5550320,)
   配置完成
   📌 使用纯 PnL 环境（HFTBacktest 原生设计）


## 🎯 PMM 强化学习环境说明

### 环境特点

我们成功构建了一个基于纯 PMM 策略(pmm_pure.py)的 TorchRL 强化学习环境，使用**纯 PnL 设计**，完全遵循 HFTBacktest 的原生理念。

#### 🎮 **动作空间** (4 维连续动作)

1. **half_spread** (1-50): 半价差（tick 数）
2. **skew** (1-50): 偏度系数（tick 数），每个标准化仓位偏移的 tick 数
3. **grid_num** (1-10): 网格订单层数
4. **grid_interval** (1-50): 网格间隔（tick 数）

**注意**:

-   `skew` 现在以 tick 数为单位，提供跨资产的标准化行为
-   `order_qty_dollar` 被设为固定值 50.0 美元，因为它主要影响资金管理而非策略核心逻辑

#### 👁️ **观测空间** (4 维状态向量)

1. **标准化中间价格**: 当前市场中间价/1000
2. **价差**: (ask-bid)/mid_price \* 1000（放大信号）
3. **标准化仓位**: 当前持仓/1000
4. **PnL**: 累计盈亏（美元）

#### 🏆 **奖励函数** (最直接设计)

-   **PnL 奖励**: 直接使用 PnL 变化（赚 1 美元奖励 1，亏 1 美元奖励-1）
-   **风险惩罚**: 基于实际仓位价值的惩罚（仓位价值超过 1000 美元开始惩罚）
-   **无交易成本**: 负费率环境下，返佣已包含在 PnL 中

#### 💡 **设计理念**

-   **无 initial_balance**: HFTBacktest 原生从 0 开始记录 PnL
-   **直接使用 PnL**: 赚多少就是多少，最直接的奖励信号
-   **基于价值的风险控制**: 使用实际仓位价值而非抽象比例
-   **KISS 原则**: 保持简单直接，避免不必要的变换

### 策略说明

使用`pmm_pure.py`中的纯 PMM 做市策略：

-   基于中间价计算基础价格
-   根据持仓风险调整保留价格（skew _ tick_size _ normalized_position）
-   在保留价格 ± 半价差位置下单做市
-   不考虑订单簿失衡(OBI)信号，专注于纯做市逻辑

### BacktestAsset 配置说明

使用链式调用配置 BacktestAsset：

```python
data_asset = (
    BacktestAsset()
    .data([data_file])              # 数据文件列表
    .linear_asset(1.0)              # 线性资产，合约乘数
    .risk_adverse_queue_model()     # 队列模型
    .no_partial_fill_exchange()     # 交易所模型
    .tick_size(0.01)               # 价格精度
    .lot_size(0.001)               # 最小交易量
    .trading_value_fee_model(-0.00003, 0.0007)  # 手续费模型
)
```


In [3]:
# 简单的训练循环示例
print("🎯 简单训练循环示例...")

try:
    # 定义动作空间边界
    # [half_spread, skew(tick数), grid_num, grid_interval_multiplier]
    action_low = [1.0, 1.0, 1.0, 1.0]
    action_high = [50.0, 50.0, 10.0, 50.0]

    max_steps = 3600  # 每个episode内策略最多执行500次
    risk_penalty_weight = 0.01
    step_interval_ns = 500_000_000  # 每次策略执行间隔1秒

    # 创建纯 PnL 环境实例
    env = create_pmm_env(
        data_asset=data_asset,
        action_low=action_low,
        action_high=action_high,
        max_steps=max_steps,
        device=device.type,
        risk_penalty_weight=risk_penalty_weight,
        step_interval_ns=step_interval_ns,  # 每次策略执行间隔1秒
    )

    # =================== Episode 和 Steps 的关系说明 ===================
    # Episode（回合）：一个完整的训练周期，从初始状态开始到结束条件触发
    # Steps（步数）：在一个episode内，策略执行的次数
    #
    # 层级关系：
    # - 训练过程包含多个episode（这里是3个）
    # - 每个episode包含多个steps（最多max_steps个）
    # - 每个step执行一次策略（调整参数、下单、等待1秒）
    #
    # 具体数字：
    # - num_episodes = 3：进行3个独立的训练回合
    # - max_steps = 500：每个episode内策略最多执行500次
    # - step_interval_ns = 1秒：每次策略执行向前推进1秒
    # - 每个episode模拟时间 = max_steps × 1秒 = 500秒（约8.3分钟）
    # - 总策略执行次数 = num_episodes × max_steps = 1500次
    # ==================================================================

    # 运行多个episode（训练回合）
    num_episodes = 3  # 运行3个独立的训练回合
    episode_rewards = []

    for episode in range(num_episodes):
        print(f"\n📍 Episode {episode + 1}/{num_episodes}")

        # ============ Episode 开始 ============
        # 每个episode都是独立的训练回合：
        # 1. 环境重置到初始状态（PnL=0，仓位=0）
        # 2. 数据从头开始读取
        # 3. 所有状态归零
        obs_td = env.reset()
        episode_reward = 0
        step_count = 0

        # ============ Episode 内的 Steps 循环 ============
        # 在这个episode中，策略会执行多次（steps）
        # 每次执行（step）包括：
        # 1. 根据当前观测选择动作（调整策略参数）
        # 2. 执行策略（下单、撤单等）
        # 3. 向前推进1秒，处理市场事件
        # 4. 获得奖励，更新状态
        while True:
            # Step 1: 生成动作（策略参数）
            # 实际训练时会被SAC策略网络的输出替代
            # 这里用随机动作演示
            random_values = torch.rand(4, device=env.device)
            action_tensor_low = torch.tensor(action_low, device=env.device)
            action_tensor_high = torch.tensor(action_high, device=env.device)

            # 生成在动作空间范围内的随机动作
            action = action_tensor_low + random_values * \
                (action_tensor_high - action_tensor_low)

            # Step 2: 执行动作（策略执行一次）
            # 这会：
            # - 更新策略参数（half_spread, skew等）
            # - 调用pure_pmm_step执行策略
            # - 向前推进1秒（step_interval_ns）
            # - 处理这1秒内的所有市场事件
            action_td = TensorDict(
                {"action": action}, batch_size=(), device=env.device
            )
            step_td = env.step(action_td)  # 策略执行一次，时间推进1秒

            # Step 3: 获取本次执行的奖励
            if 'reward' in step_td:
                reward = step_td['reward'].item()
            elif 'next' in step_td and 'reward' in step_td['next']:
                reward = step_td['next']['reward'].item()
            else:
                reward = 0

            episode_reward += reward  # 累计本episode的总奖励
            step_count += 1  # 策略执行次数+1

            # Step 4: 检查episode是否结束
            # 结束条件：
            # - 数据用完（最常见）
            # - 达到max_steps
            # - PnL损失超过阈值
            if 'done' in step_td:
                done = step_td['done'].item()
                next_obs = step_td.get('observation', obs_td['observation'])
            elif 'next' in step_td:
                done = step_td['next']['done'].item(
                ) if 'done' in step_td['next'] else False
                next_obs = step_td['next'].get(
                    'observation', obs_td['observation'])
            else:
                done = False
                next_obs = obs_td['observation']

            if done:
                break  # 本episode结束

            # Step 5: 更新观测，继续下一个step
            obs_td = TensorDict(
                {"observation": next_obs}, batch_size=(), device=env.device
            )
            # 循环继续，执行下一次策略（下一个step）

        # ============ Episode 结束 ============
        episode_rewards.append(episode_reward)

        # 获取最终状态并显示统计
        final_state = env._get_strategy_state()
        final_pnl = final_state['pnl']  # 直接的 PnL，从 0 开始

        print(f"  步数: {step_count}")  # 本episode内策略执行了多少次
        print(f"  模拟时长: {step_count}秒 ({step_count/60:.1f}分钟)")  # 模拟了多长时间的市场
        print(f"  总奖励: {episode_reward:.4f}")
        print(f"  最终PnL: ${final_pnl:.2f}")
        print(f"  回报率: {final_pnl / 50.0:.8%}")  # 基于50美元订单金额的回报率

    # ============ 所有Episodes完成 ============
    # 训练总结
    print(f"\n📊 训练总结:")
    print(f"  运行了 {num_episodes} 个episode")
    print(f"  每个episode最多 {max_steps} 个steps（策略执行次数）")
    print(f"  总策略执行次数: 约{num_episodes * max_steps}次")
    print(f"  平均奖励: {sum(episode_rewards) / len(episode_rewards):.4f}")
    print(f"  最佳奖励: {max(episode_rewards):.4f}")
    print(f"  最差奖励: {min(episode_rewards):.4f}")

    env.close()
    print("\n✅ 训练循环示例完成!")

except Exception as e:
    print(f"❌ 训练循环失败: {e}")
    import traceback
    traceback.print_exc()

🎯 简单训练循环示例...

📍 Episode 1/3
  步数: 3600
  模拟时长: 3600秒 (60.0分钟)
  总奖励: 0.0000
  最终PnL: $0.00
  回报率: 0.00000000%

📍 Episode 2/3
  步数: 3600
  模拟时长: 3600秒 (60.0分钟)
  总奖励: 0.0000
  最终PnL: $0.00
  回报率: 0.00000000%

📍 Episode 3/3
  步数: 3600
  模拟时长: 3600秒 (60.0分钟)
  总奖励: 0.0000
  最终PnL: $0.00
  回报率: 0.00000000%

📊 训练总结:
  运行了 3 个episode
  每个episode最多 3600 个steps（策略执行次数）
  总策略执行次数: 约10800次
  平均奖励: 0.0000
  最佳奖励: 0.0000
  最差奖励: 0.0000

✅ 训练循环示例完成!
